# Проверочные расчёты к статье «Теорема Нётер и сотовый двигатель Казимира»

Ноутбук собирает все численные проверки, на которые ссылается текст `noether_casimir_drive_ru.tex` (и его новая версия `_v2`).
Каждая выкладка: сначала Markdown с формулами, затем ячейка кода, которая печатает `OK`/`FAIL` и невязку.

| § | что проверяется | где в статье |
|---|---|---|
| 1 | поля TM/TE мод удовлетворяют уравнениям Максвелла и граничным условиям (символьно) | Прил. А, п. 2–3 |
| 2 | то же численно для моды $(2,1,3)$, $a:b:h=1:1{,}3:2{,}7$ | Прил. А, п. 3 |
| 3 | $p=-\langle T_{zz}\rangle=\overline w\,k_z^2/k^2$, независимость от $z$ | разд. 3 |
| 4 | ошибка Прил. B работы 2024 г.: $0{,}2119$ вместо $1$ | разд. 3 |
| 5 | помодовое тождество $-\partial_h(\hbar ck/2)=S\,p$ | разд. 3 |
| 6 | скачок на металле: $-\langle T_{zz}\rangle(0^+)=p$ | разд. 4 |
| 7 | две пластины: $p_{in}-p_{out}\to-\pi^2/240$ | разд. 6 |
| 8 | регуляризованные интегралы $G$, $G_T$ (замкнутая форма) | Прил. Б |
| 9 | пересчёт табл. 1 работы 2024 г. | разд. 6 |
| 10 | тензорная сумма $\sum k_z^2/k$ против энергетической $\sum k$ | разд. 6, 7 |
| 11 | оценка тяги для $a=2$ мкм | разд. 6 |

Ядро: SageMath 9.6. Единицы гауссовы, $c=1$ там, где не оговорено.

In [ ]:
import numpy as np
np.set_printoptions(precision=10)
def check(name, residual, tol):
    residual = float(abs(residual))
    print(('OK  ' if residual < tol else 'FAIL') + '  %-60s residual = %.3e (tol %.0e)' % (name, residual, tol))
    return residual < tol
def trap(y, x):
    y = np.asarray(y); x = np.asarray(x)
    return float(np.sum((y[1:] + y[:-1]) * np.diff(x)) / 2.0)

## 1. Поля мод: символьная проверка

Резонатор $a\times b\times h$, $k_x=\pi n_x/a$, $k_y=\pi n_y/b$, $k_z=\pi n_z/h$, $K^2=k_x^2+k_y^2$, $k^2=K^2+k_z^2$, $\omega=ck$.

**TM** ($H_z=0$):
$E_z=A\sin k_xx\sin k_yy\cos k_zz\cos\omega t$,
$E_x=-A\frac{k_xk_z}{K^2}\cos k_xx\sin k_yy\sin k_zz\cos\omega t$,
$E_y=-A\frac{k_yk_z}{K^2}\sin k_xx\cos k_yy\sin k_zz\cos\omega t$,
$H_x=-A\frac{k\,k_y}{K^2}\sin k_xx\cos k_yy\cos k_zz\sin\omega t$,
$H_y=A\frac{k\,k_x}{K^2}\cos k_xx\sin k_yy\cos k_zz\sin\omega t$.

**TE** ($E_z=0$): то же с заменой $E\leftrightarrow H$, $\sin k_zz\leftrightarrow\cos k_zz$, $\sin k_{x,y}\leftrightarrow\cos k_{x,y}$ (см. статью).

Проверяем: $\nabla\!\cdot\!\mathbf E=\nabla\!\cdot\!\mathbf H=0$, $\nabla\times\mathbf E=-\frac1c\partial_t\mathbf H$, $\nabla\times\mathbf H=\frac1c\partial_t\mathbf E$,
и на всех шести гранях касательное $\mathbf E=0$, нормальное $\mathbf H=0$.

In [ ]:
var('x y z t a b h c A')
nx_, ny_, nz_ = var('n_x n_y n_z')
assume(a>0, b>0, h>0, c>0)
kx = pi*nx_/a; ky = pi*ny_/b; kz = pi*nz_/h
K2 = kx^2 + ky^2; k = sqrt(K2 + kz^2); w = c*k

def maxwell_residuals(E, H):
    divE = sum(diff(E[i], v) for i, v in enumerate((x, y, z)))
    divH = sum(diff(H[i], v) for i, v in enumerate((x, y, z)))
    curl = lambda F: (diff(F[2], y) - diff(F[1], z), diff(F[0], z) - diff(F[2], x), diff(F[1], x) - diff(F[0], y))
    cE, cH = curl(E), curl(H)
    r1 = [ (cE[i] + diff(H[i], t)/c).simplify_full() for i in range(3)]
    r2 = [ (cH[i] - diff(E[i], t)/c).simplify_full() for i in range(3)]
    return [divE.simplify_full(), divH.simplify_full()] + r1 + r2

def bc_residuals(E, H):
    sub_int = {nx_: 2, ny_: 3, nz_: 5}   # целые индексы: sin(pi*n)=0
    out = []
    for (v, L, tang, norm) in [(x, a, (1, 2), 0), (y, b, (0, 2), 1), (z, h, (0, 1), 2)]:
        for val in (0, L):
            for i in tang: out.append(E[i].subs({v: val}).subs(sub_int).simplify_full())
            out.append(H[norm].subs({v: val}).subs(sub_int).simplify_full())
    return out

TM_E = (-A*kx*kz/K2*cos(kx*x)*sin(ky*y)*sin(kz*z)*cos(w*t),
        -A*ky*kz/K2*sin(kx*x)*cos(ky*y)*sin(kz*z)*cos(w*t),
         A*sin(kx*x)*sin(ky*y)*cos(kz*z)*cos(w*t))
TM_H = (-A*k*ky/K2*sin(kx*x)*cos(ky*y)*cos(kz*z)*sin(w*t),
         A*k*kx/K2*cos(kx*x)*sin(ky*y)*cos(kz*z)*sin(w*t), 0)
TE_H = (-A*kx*kz/K2*sin(kx*x)*cos(ky*y)*cos(kz*z)*cos(w*t),
        -A*ky*kz/K2*cos(kx*x)*sin(ky*y)*cos(kz*z)*cos(w*t),
         A*cos(kx*x)*cos(ky*y)*sin(kz*z)*cos(w*t))
TE_E = (-A*k*ky/K2*cos(kx*x)*sin(ky*y)*sin(kz*z)*sin(w*t),
         A*k*kx/K2*sin(kx*x)*cos(ky*y)*sin(kz*z)*sin(w*t), 0)

for name, E, H in [('TM', TM_E, TM_H), ('TE', TE_E, TE_H)]:
    rm = maxwell_residuals(E, H); rb = bc_residuals(E, H)
    print(name, 'Maxwell:', 'OK' if all(r == 0 for r in rm) else ('FAIL', rm))
    print(name, 'граничные условия:', 'OK' if all(r == 0 for r in rb) else ('FAIL', rb))

## 2. Численная проверка моды $(2,1,3)$ при $a:b:h=1:1{,}3:2{,}7$

Производные — центральными разностями в случайных точках резонатора; в статье заявлена невязка $\sim10^{-9}$.
Здесь же печатается отношение $k_z^2/k^2$, которое в п. 12(а) статьи приведено как $0{,}2119$.

In [ ]:
rng = np.random.default_rng(1)
A_, a_, b_, h_ = 1.0, 1.0, 1.3, 2.7
n = (2, 1, 3)
Kx, Ky, Kz = np.pi*n[0]/a_, np.pi*n[1]/b_, np.pi*n[2]/h_
KK = Kx**2 + Ky**2; kk = np.sqrt(KK + Kz**2); om = kk   # c = 1

def TM(p, tt):
    X, Y, Z = p
    E = np.array([-A_*Kx*Kz/KK*np.cos(Kx*X)*np.sin(Ky*Y)*np.sin(Kz*Z)*np.cos(om*tt),
                  -A_*Ky*Kz/KK*np.sin(Kx*X)*np.cos(Ky*Y)*np.sin(Kz*Z)*np.cos(om*tt),
                   A_*np.sin(Kx*X)*np.sin(Ky*Y)*np.cos(Kz*Z)*np.cos(om*tt)])
    H = np.array([-A_*kk*Ky/KK*np.sin(Kx*X)*np.cos(Ky*Y)*np.cos(Kz*Z)*np.sin(om*tt),
                   A_*kk*Kx/KK*np.cos(Kx*X)*np.sin(Ky*Y)*np.cos(Kz*Z)*np.sin(om*tt), 0.0*X])
    return E, H
def TE(p, tt):
    X, Y, Z = p
    H = np.array([-A_*Kx*Kz/KK*np.sin(Kx*X)*np.cos(Ky*Y)*np.cos(Kz*Z)*np.cos(om*tt),
                  -A_*Ky*Kz/KK*np.cos(Kx*X)*np.sin(Ky*Y)*np.cos(Kz*Z)*np.cos(om*tt),
                   A_*np.cos(Kx*X)*np.cos(Ky*Y)*np.sin(Kz*Z)*np.cos(om*tt)])
    E = np.array([-A_*kk*Ky/KK*np.cos(Kx*X)*np.sin(Ky*Y)*np.sin(Kz*Z)*np.sin(om*tt),
                   A_*kk*Kx/KK*np.sin(Kx*X)*np.cos(Ky*Y)*np.sin(Kz*Z)*np.sin(om*tt), 0.0*X])
    return E, H

def residual(mode, eps=1e-4, npts=50):
    worst = 0.0
    for _ in range(npts):
        p = rng.uniform([0, 0, 0], [a_, b_, h_]); tt = rng.uniform(0, 2*np.pi/om)
        J = {}
        for name, i in (('E', 0), ('H', 1)):
            D = np.zeros((3, 3))
            for j in range(3):
                d = np.zeros(3); d[j] = eps
                D[:, j] = (mode(p + d, tt)[i] - mode(p - d, tt)[i]) / (2*eps)
            J[name] = D
        dt = lambda i: (mode(p, tt + eps)[i] - mode(p, tt - eps)[i]) / (2*eps)
        curl = lambda D: np.array([D[2, 1] - D[1, 2], D[0, 2] - D[2, 0], D[1, 0] - D[0, 1]])
        r = [np.trace(J['E']), np.trace(J['H']),
             *(curl(J['E']) + dt(1)), *(curl(J['H']) - dt(0))]
        worst = max(worst, np.max(np.abs(r)))
    return worst

for nm, md_ in (('TM', TM), ('TE', TE)):
    check('Maxwell, мода (2,1,3), ' + nm + ' (разностная схема O(eps^2))', residual(md_), 1e-6)
print('k_z^2/k^2 =', Kz**2/kk**2, ' (в статье 0,2119)')
check('k_z^2/k^2 = 0.2119', Kz**2/kk**2 - 0.2119, 1e-4)

## 3. $p=-\langle T_{zz}\rangle=\overline w\,k_z^2/k^2$ и независимость от $z$

$T_{zz}=\frac1{8\pi}(E_z^2+H_z^2-E_x^2-E_y^2-H_x^2-H_y^2)$. Усредняем по сечению $ab$ и по периоду (точными квадратурами по средним точкам)
в нескольких плоскостях $z$ и сравниваем с $\overline w\,k_z^2/k^2$, где $\overline w$ — плотность энергии, усреднённая по всему объёму.

In [ ]:
def avg_T(mode, Z, N=64, Nt=64):
    X = (np.arange(N) + 0.5) / N * a_; Y = (np.arange(N) + 0.5) / N * b_
    T = (np.arange(Nt) + 0.5) / Nt * 2*np.pi/om
    XX, YY, TT = np.meshgrid(X, Y, T, indexing='ij'); ZZ = np.full_like(XX, Z)
    E, H = mode((XX, YY, ZZ), TT)
    Tzz = (E[2]**2 + H[2]**2 - E[0]**2 - E[1]**2 - H[0]**2 - H[1]**2) / (8*np.pi)
    w = (np.sum(E**2, 0) + np.sum(H**2, 0)) / (8*np.pi)
    return Tzz.mean(), w.mean()

for nm, md_ in (('TM', TM), ('TE', TE)):
    Zs = (np.arange(64) + 0.5) / 64 * h_
    wbar = np.mean([avg_T(md_, Z)[1] for Z in Zs])
    vals = np.array([-avg_T(md_, Z)[0] for Z in (0.0, 0.37*h_, 0.81*h_, h_)])
    target = wbar * Kz**2 / kk**2
    check(nm + ': -<T_zz>(z) = wbar k_z^2/k^2 при z=0, .37h, .81h, h', np.max(np.abs(vals - target)) / target, 1e-10)

## 4. Ошибка в Приложении B работы 2024 г.

Там давление на грань $xy$ вычислялось как $\langle H_x^2+H_y^2\rangle/8\pi$ (без $-\langle E_z^2\rangle/8\pi$). Для TM-моды это даёт отношение
$(f_z/S)/(E/V)=1$; с полным $T_{zz}$ на стенке получается $k_z^2/k^2$. Для TE-мод $E_z\equiv0$, и обе формулы совпадают.

In [ ]:
for nm, md_ in (('TM', TM), ('TE', TE)):
    Zs = (np.arange(64) + 0.5) / 64 * h_
    wbar = np.mean([avg_T(md_, Z)[1] for Z in Zs])
    N = 64; X = (np.arange(N)+.5)/N*a_; Y = (np.arange(N)+.5)/N*b_; T = (np.arange(N)+.5)/N*2*np.pi/om
    XX, YY, TT = np.meshgrid(X, Y, T, indexing='ij')
    E, H = md_((XX, YY, 0*XX), TT)
    old = np.mean(H[0]**2 + H[1]**2) / (8*np.pi) / wbar
    new = np.mean(H[0]**2 + H[1]**2 - E[2]**2) / (8*np.pi) / wbar
    print('%s:  формула 2024 г. -> %.4f,   полный T_zz -> %.4f,   k_z^2/k^2 = %.4f' % (nm, old, new, Kz**2/kk**2))
Zs = (np.arange(64) + 0.5) / 64 * h_
wbar = np.mean([avg_T(TM, Z)[1] for Z in Zs])
E, H = TM((XX, YY, 0*XX), TT)
check('TM: H_t^2/8pi / wbar = 1 (результат прил. B 2024)', np.mean(H[0]**2+H[1]**2)/(8*np.pi)/wbar - 1, 1e-10)
check('TM: (H_t^2 - E_z^2)/8pi / wbar = k_z^2/k^2', np.mean(H[0]**2+H[1]**2-E[2]**2)/(8*np.pi)/wbar - Kz**2/kk**2, 1e-10)

## 5. Помодовое тождество (Гельман–Фейнман)

При $k_z=\pi n_z/h$: $\partial k/\partial h=-k_z^2/(kh)$, поэтому $-\dfrac{\partial}{\partial h}\dfrac{\hbar ck}{2}=\dfrac{\hbar c}{2h}\dfrac{k_z^2}{k}=S\,p_{\text{моды}}$, $V=Sh$.
Проверяем производную численно для набора мод.

In [ ]:
ok = True; worst = 0
for (n1, n2, n3) in [(1,1,1), (2,1,3), (0,3,2), (5,2,7), (1,0,1)]:
    kf = lambda hh: np.sqrt((np.pi*n1/a_)**2 + (np.pi*n2/b_)**2 + (np.pi*n3/hh)**2)
    e = 1e-6
    lhs = -(kf(h_+e) - kf(h_-e)) / (2*e) / 2         # -d/dh (k/2), hbar c = 1
    kz_ = np.pi*n3/h_
    rhs = kz_**2 / kf(h_) / (2*h_)
    worst = max(worst, abs(lhs - rhs) / rhs)
check('-d/dh (hbar c k/2) = (hbar c/2h) k_z^2/k', worst, 1e-8)

## 6. Скачок $T_{zz}$ на поверхности металла

Со стороны вакуума при $z=0^+$: $E_x=E_y=H_z=0$, так что $T_{zz}=\frac1{8\pi}(E_z^2-H_x^2-H_y^2)$; в металле $T_{zz}=0$.
Скачок $[\![T_{zz}]\!]=p$, то есть то же значение, что в объёме. Разбиение на силу на заряды $-\langle E_n^2\rangle/8\pi$ и на токи $\langle H_t^2\rangle/8\pi$:
у TE-мод зарядов нет вовсе.

In [ ]:
Zs = (np.arange(64) + 0.5) / 64 * h_
for nm, md_ in (('TM', TM), ('TE', TE)):
    wbar = np.mean([avg_T(md_, Z)[1] for Z in Zs])
    E, H = md_((XX, YY, 0*XX), TT)
    fq = -np.mean(E[2]**2)/(8*np.pi); fi = np.mean(H[0]**2 + H[1]**2)/(8*np.pi)
    print('%s: сила на заряды %.5f, на токи %.5f, сумма %.5f, wbar k_z^2/k^2 = %.5f' % (nm, fq, fi, fq+fi, wbar*Kz**2/kk**2))
    check(nm + ': [[T_zz]] = p', (fq + fi) - wbar*Kz**2/kk**2, 1e-12)

## 7. Две пластины Казимира

В зазоре $a$ (две поляризации, $\hbar c=1$, $a=1$): $p_{in}=\sum_{n\ge1}\int\frac{d^2k_\parallel}{(2\pi)^2}\frac{k_z^2}{k}e^{-k/k_m}$,
снаружи сумма заменяется интегралом. Интеграл по $k_\parallel$ берётся явно: $\int_0^\infty \frac{q\,dq}{2\pi}\frac{k_z^2}{k}e^{-k/k_m}=\frac{k_z^2 k_m}{2\pi}e^{-k_z/k_m}$.
Ожидается $p_{in}-p_{out}\to-\pi^2/240=-0{,}041123$; в статье: $-0{,}041043,\,-0{,}041110,\,-0{,}041120$ при $k_m a=20,50,100$.

In [ ]:
paper = {20: -0.041043, 50: -0.041110, 100: -0.041120}
for km in (20, 50, 100, 400):
    nn = np.arange(1, 80*km)
    kz_ = np.pi * nn
    p_in = np.sum(kz_**2 * km / (2*np.pi) * np.exp(-kz_/km))
    p_out = (1/np.pi) * km / (2*np.pi) * 2 * km**3          # (a/pi) int dk_z k_z^2 k_m e^{-k_z/k_m}/(2pi)
    d = p_in - p_out
    s = '' if km not in paper else '   (в статье %.6f)' % paper[km]
    print('k_m a = %4d:  p_in - p_out = %.6f%s' % (km, d, s))
    if km in paper: check('совпадение со статьёй, k_m a=%d' % km, d - paper[km], 2e-6)
check('предел -pi^2/240', d + np.pi**2/240, 1e-6)

## 8. Регуляризованные модовые интегралы (Прил. А работы 2024 г.)

$f(k/k_m)=1/(1+k^4/k_m^4)$, $x\equiv ak_m$, $c=(\pi/x)^4$, $m=\sqrt{n_x^2+n_y^2}$.

Энергия: $G(m)=\int_0^\infty\sqrt{m^2+u^2}\,f\,du=\int_m^\infty\frac{n^2 f(n)}{\sqrt{n^2-m^2}}dn$.

Тензор: $G_T(m)=\int_0^\infty\frac{u^2}{\sqrt{m^2+u^2}}\,f\,du=G(m)-m^2H(m)$, $H(m)=\int_m^\infty\frac{f(n)\,dn}{\sqrt{n^2-m^2}}$.

С $g=-i/\sqrt c$ и $J(m^2)=\dfrac{\operatorname{artanh}\sqrt{g/(m^2+g)}}{\sqrt{g/(m^2+g)}\,(m^2+g)}$ имеем $G=\operatorname{Re}J/c$, $H=\operatorname{Re}(-iJ/\sqrt c)$, $G(0)=\pi/(4\sqrt c)$.
Проверяем замкнутую форму квадратурой с подстановкой $n=m\cosh s$.

In [ ]:
def GG(m, xk):
    cc = (np.pi/xk)**4; g = -1j/np.sqrt(cc); m2 = np.asarray(m, dtype=float)**2
    with np.errstate(all='ignore'):
        wv = np.sqrt(g/(m2 + g)); Jv = np.arctanh(wv)/(wv*(m2 + g))
        G = (Jv/cc).real; H = ((-1j/np.sqrt(cc))*Jv).real
    G = np.where(m2 == 0, np.pi/(4*np.sqrt(cc)), G)
    GT = np.where(m2 == 0, G, G - m2*H)
    return G, GT

worst = 0
for xk in (1.0, 4.0, 9.0):
    cc = (np.pi/xk)**4
    for m in (0.3, 2.0, 7.0):
        s = np.linspace(0, 14, 400001); nn = m*np.cosh(s)
        Gq = trap(m**2*np.cosh(s)**2/(1 + cc*nn**4), s)
        GTq = trap(m**2*np.sinh(s)**2/(1 + cc*nn**4), s)
        G, GT = GG(m, xk)
        worst = max(worst, abs(G - Gq)/Gq, abs(GT - GTq)/GTq)
check('замкнутая форма G, G_T против квадратуры', worst, 1e-7)

## 9. Пересчёт табл. 1 работы 2024 г.

$D_E(x)=\sum_{n_x,n_y=(0)1}^{\infty}G-\int_0^\infty\!\!\int_0^\infty G\,dn_xdn_y$, где запись $(0)1$ означает вес $\tfrac12$ на осях и $\tfrac14$ в углу
(формула (13) работы 2024 г.). Считаем на квадрате $[0,N]^2$: сумма с трапециевидными весами минус интеграл (Гаусс по 10 точек в каждой ячейке),
затем экстраполируем по $N\to\infty$ (погрешность обрезания $\propto1/N^2$, Ричардсон).

Время: около минуты на точку при $N=600$.

In [ ]:
gx, gw = np.polynomial.legendre.leggauss(10); gx = (gx + 1)/2; gw = gw/2
def D(xk, N, col=0):
    nn = np.arange(N + 1.0); M = np.sqrt(nn[:, None]**2 + nn[None, :]**2)
    w = np.ones(N + 1); w[0] = w[-1] = 0.5
    S = float(np.sum(w[:, None]*w[None, :]*GG(M, xk)[col]))
    p = (np.arange(N)[:, None] + gx[None, :]).ravel(); pw = np.tile(gw, N)
    I = 0.0
    for i in range(0, len(p), 400):
        Gm = GG(np.sqrt(p[i:i+400, None]**2 + p[None, :]**2), xk)[col]
        I += float(np.sum(pw[i:i+400, None]*pw[None, :]*Gm))
    return S - I
def D_inf(xk, col=0, N1=300, N2=600):
    d1, d2 = D(xk, N1, col), D(xk, N2, col)
    return d2 + (d2 - d1)/((N2/N1)**2 - 1)   # ошибка обрезания ~ 1/N^2 (проверено: разности падают в 4 раза)

table2024 = {0.25: 0.00108989, 0.5: 0.00322377, 1: 0.00304837, 1.5: -0.00870823, 2: -0.0251746,
             2.5: -0.0332867, 3: -0.0289864, 4: -0.00973151, 5: -0.00732334, 6: -0.0124360, 9: -0.0137827}
DE = {}
print(' a k_m     табл.2024     пересчёт     отклонение')
for xk, v in table2024.items():
    DE[xk] = D_inf(xk)
    print('%6.2f  %12.8f  %12.8f   %6.1f %%' % (xk, v, DE[xk], 100*(v - DE[xk])/abs(DE[xk])))
check('совпадение при a k_m <= 2.5 (лучше 1%)', max(abs(table2024[x_]-DE[x_])/abs(DE[x_]) for x_ in table2024 if x_ <= 2.5), 1e-2)

**Вывод.** При $ak_m\lesssim3$ таблица 2024 г. воспроизводится. При больших $ak_m$ пересчёт даёт $D_E\approx-0{,}0120$ (и $-0{,}0121$ при $ak_m=14$), а табличное значение при $ak_m=9$ ($-0{,}01378$) завышено по модулю на $\approx15\%$. Вероятная причина — недостаточная область суммирования/интегрирования при большом $ak_m$ в исходном расчёте 2024 г. (здесь сходимость по $N$ контролируется явно). Именно это значение использовано для коэффициента $0{,}0136$ и оценки $8{,}55$ дин/м².

## 10. Тензорная сумма против энергетической

Сила через тензор натяжений: $D_T(x)=\sum_{(0)1}G_T-\iint G_T$ (сумма $\sum k_z^2/k$ с тем же регулятором). Через энергию: $D_E$.
Для полого волновода (ячейка сот) энергия линейна по длине, и по буст‑аргументу (Браун–Маклей) $\langle\Pi_{zz}\rangle=-\langle u\rangle$, поэтому
ожидается $D_T\to-D_E$ при $ak_m\to\infty$. При конечном $ak_m$ разница — это член с $f'$ из п. 12(б) статьи.
В черновике статьи указано, что $\sum k_z^2/k$ для сотовой пластины «ещё не вычислена» — вычисляем.

In [ ]:
print(' a k_m      D_E          D_T        D_T + D_E')
DT = {}
for xk in (1, 2.5, 5, 9, 14):
    N1, N2 = (400, 800) if xk >= 9 else (300, 600)
    DE[xk] = D_inf(xk, 0, N1, N2)
    DT[xk] = D_inf(xk, 1, N1, N2)
    print('%6.1f  %11.7f  %11.7f  %11.7f' % (xk, DE[xk], DT[xk], DT[xk] + DE[xk]))
check('D_T + D_E -> 0 при a k_m = 14 (относительно |D_E|)', (DT[14] + DE[14])/abs(DE[14]), 1e-2)
R_inf = 0.5*(DT[14] - DE[14])
print('R(a k_m -> oo) ~ %.5f' % R_inf)

**Вывод.** При малых $ak_m$ тензорная и энергетическая суммы различаются даже знаком (член с $f'$), но при $ak_m\to\infty$ сходятся к одному
пределу $|R|\approx0{,}0121$ ($D_T+D_E=4\cdot10^{-6}$ при $ak_m=14$): тензорный и гамильтонов выводы дают одну и ту же силу, как и требует помодовое тождество §5 плюс буст‑аргумент.

## 11. Оценка тяги

$F/S=\hbar c\,\pi R/a^4$. $\hbar c=3{,}16153\times10^{-17}$ эрг·см. Для $a$ в микронах $\hbar c\pi/a^4=0{,}99322/a_\mu^4$ дин/см².

In [ ]:
hbar_c = 3.16152677e-17          # erg*cm
coef = hbar_c*np.pi/(1e-4)**4     # dyn/cm^2 при a = 1 мкм
check('hbar c pi / (1 мкм)^4 = 0.99322 дин/см^2', coef - 0.99322, 1e-5)
for name, R in (('R = 0.0137827 (табл. 2024, a k_m = 9)', 0.0137827), ('R = %.5f (пересчёт, предел)' % abs(R_inf), abs(R_inf))):
    Fs = coef*R/2**4*1e4
    print('%-40s  0.0136-коэффициент -> %.5f;  F/S(a=2 мкм) = %.2f дин/м^2' % (name, coef*R, Fs))
check('8.55 дин/м^2 воспроизводится из R=0.0137827', coef*0.0137827/16*1e4 - 8.55, 0.01)

## Итог

* Поля мод, граничные условия, $p=\overline w\,k_z^2/k^2$, независимость от $z$, скачок на металле, помодовое тождество и две пластины — **подтверждены**.
* Ошибка Прил. B 2024 г. **подтверждена**: для TM‑моды $1$ вместо $k_z^2/k^2$ ($0{,}2119$ для моды $(2,1,3)$). На гамильтонов результат она не влияет.
* Тензорная сумма для сот **вычислена**: при $ak_m\to\infty$ $D_T=-D_E$, то есть $|F/S|=\delta(E/V)$ — силы совпадают.
* Табл. 1 2024 г. **расходится** с пересчётом при $ak_m\gtrsim4$; предел $R\approx0{,}0121$ даёт $\approx7{,}5$ дин/м² вместо $8{,}55$ для $a=2$ мкм. Это стоит перепроверить вашим исходным кодом.